# Software profesional en Acústica 2025-26 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/).  The FEniCS implementation has been replaced with NGSolve. The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [ ]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Revisiting the numerical solution of the acoustic model: Raviart-Thomas elements and the displacement formulation

This notebook illustrates the numerical solution of the wave equation for harmonic excitation using the so called [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM). The method aims at an approximate solution by subdividing the area of interest into smaller parts with simpler geometry, linking these parts together and applying methods from the calculus of variations to solve the problem numerically. The FEM is a well established method for the numerical approximation of the solution of partial differential equations (PDEs). The solutions of PDEs are often known analytically only for rather simple geometries. FEM based simulations allow to gain insights into other more complex cases.

## Problem Statement

The acoustic model of a compressible fluid can also be written in terms of the displacement field, given as

\begin{equation*}
c^2 \nabla \mathrm{div} \boldsymbol{u}(\boldsymbol{x}, t) - \frac{\partial^2}{\partial t^2} \boldsymbol{u}(\boldsymbol{x}, t) = \boldsymbol{0},
\end{equation*}

where $\boldsymbol{u}(\boldsymbol{x}, t)$ denotes the particle displacement at position $\boldsymbol{x}$, $c$ the speed of sound.
For an harmonic excitation, we choose the Ansatz $\boldsymbol{u}(\boldsymbol{x}, t) = \Re \{ \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{e}^{-\mathrm{i} \omega t} \}$ for the sound pressure with $\omega = 2 \pi f$ 
Introduction of the complex quantities into the wave equation yields

\begin{equation*}
c^2 \nabla \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{e}^{-\mathrm{i} \omega t} + \omega^2\boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{e}^{-\mathrm{i} \omega t} = \boldsymbol{0},
\end{equation*}

and canceling out the $\mathrm{e}^{-\mathrm{i} \omega t}$ terms yields the displacement model

\begin{equation*}
c^2 \nabla \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) + \omega^2\boldsymbol{U}(\boldsymbol{x}, \omega) = \boldsymbol{0} .
\end{equation*}

We aim for a numerical solution of this model in the domain $\Omega$ with respect to a Dirichlet boundary condition
\begin{equation*}
\boldsymbol{U}(\boldsymbol{x}, \omega)\cdot \boldsymbol{n} = U_{D} \qquad \text{for } x \in \partial \Omega
\end{equation*}

where $P_{D}$ is the Dirichlet data associated with the normal displacement on the boundary.

## Variational Formulation

The FEM is based on expressing the partial differential equation (PDE) to be solved in its variational or weak form.
The first step towards this formulation is to multiply the Helmholtz equation by the test function $\boldsymbol{V}(\boldsymbol{x})$

\begin{equation*}
c^2 \nabla \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \cdot \boldsymbol{V}(\boldsymbol{x})  + \omega^2\boldsymbol{U}(\boldsymbol{x}, \omega) \cdot \boldsymbol{V}(\boldsymbol{x}) = 0,
\end{equation*}

followed by integration over the domain $\Omega$

\begin{equation*}
\int_\Omega c^2 \nabla \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \cdot \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x}  + \omega^2\int_\Omega \boldsymbol{U} \cdot \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x} = 0,
\end{equation*}

where $\mathrm{d}x$ denotes a suitably chosen differential element for integration.
Application of a classical Green's formula yields to

\begin{equation*}
{-} \int_\Omega \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{div} \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x}  + \int_{\partial \Omega} \boldsymbol{V}(\boldsymbol{x})\cdot \boldsymbol{n}\ \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{d}s + \omega^2 \int_\Omega \boldsymbol{U}(\boldsymbol{x}, \omega)\cdot \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x} = 0.
\end{equation*}

This way the differential order of the first integral is lowered which is advisable for application of the FEM.
The second integral vanishes as the variation formulation requires $V(\boldsymbol{x})\cdot\boldsymbol{n} = 0$ on $\partial \Omega$ where $(\boldsymbol{x}, \omega)$ is known - here by the pure Dirichlet boundary condition 

This results in the variational/weak formulation of the displacement model

\begin{equation*}
{-} \int_\Omega \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{div} \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x}  + \omega^2 \int_\Omega \boldsymbol{U}(\boldsymbol{x}, \omega)\cdot \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x} = 0
\end{equation*}

It is common to express the integral equation above in terms of the bilinear $a(P, V)$ and linear $L(V)$ forms 

\begin{equation*}
a(\boldsymbol{U}, \boldsymbol{V}) = \int_\Omega \mathrm{div} \boldsymbol{U}(\boldsymbol{x}, \omega) \mathrm{div} \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x}  - \omega^2 \int_\Omega \boldsymbol{U}(\boldsymbol{x}, \omega)\cdot \boldsymbol{V}(\boldsymbol{x}) \,\mathrm{d}\boldsymbol{x},
\end{equation*}

\begin{equation*}
L(\boldsymbol{V}) = 0,
\end{equation*}

where

\begin{equation*}
a(\boldsymbol{U}, \boldsymbol{V}) = L(\boldsymbol{V}) .
\end{equation*}

## Numerical Solution

The numerical solution of the variational problem is based on [NGSolve](https://ngsolve.org/), an open-source framework for numerical solution of PDEs.
The implementation is based on the variational formulation derived above.
It is common in the FEM to denote the solution of the problem by $u$ and the test function by $v$.
We limit ourselves to real-valued $\boldsymbol{U}(\boldsymbol{x}, \omega)$ due to the assumption of Dirichlet real data.

A function is defined for this purpose, accompanied by a plotting routine for the resulting sound field.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from netgen.geom2d import SplineGeometry
from ngsolve import *
from ngsolve.webgui import Draw


def make_rect_mesh(nx, ny, Lx=5.0, Ly=4.0):
    """Create a structured rectangular mesh [0,Lx] x [0,Ly] using NGSolve/Netgen."""
    geo = SplineGeometry()
    p1 = geo.AppendPoint(0,  0)
    p2 = geo.AppendPoint(Lx, 0)
    p3 = geo.AppendPoint(Lx, Ly)
    p4 = geo.AppendPoint(0,  Ly)
    geo.Append(["line", p1, p2], bc="bottom")
    geo.Append(["line", p2, p3], bc="right")
    geo.Append(["line", p3, p4], bc="top")
    geo.Append(["line", p4, p1], bc="left")
    # maxh controls element size; approximate it from nx, ny
    maxh = min(Lx / nx, Ly / ny)
    mesh = Mesh(geo.GenerateMesh(maxh=maxh))
    return mesh


def FEM_displacements(mesh, frequency, uex_cf, c=343):
    """Numerical solution of the displacement Helmholtz equation using
    first-order Raviart-Thomas (RT) elements in NGSolve.

    Parameters
    ----------
    mesh       : NGSolve Mesh object
    frequency  : excitation frequency [Hz]
    uex_cf     : CoefficientFunction (vector) with the exact / Dirichlet data
    c          : speed of sound [m/s]

    Returns
    -------
    sol : GridFunction containing the FE solution
    V   : the HDiv finite element space used
    """
    omega = 2 * np.pi * frequency
    k2 = (omega / c) ** 2

    # First-order Raviart-Thomas space (HDiv in NGSolve)
    V = HDiv(mesh, order=1, dirichlet="bottom|top|left|right", RT=True, complex=True)

    u, v = V.TnT()   # trial and test functions

    # Bilinear form:  a(u,v) = (div u, div v) - k^2 (u, v)
    a = BilinearForm(V)
    a += div(u) * div(v) * dx - k2 * InnerProduct(u, v) * dx
    a.Assemble()

    # Linear form: L(v) = 0
    f = LinearForm(V)
    f.Assemble()

    # Apply Dirichlet boundary condition via the normal trace
    sol = GridFunction(V)
    sol.Set(uex_cf, definedon=mesh.Boundaries("bottom|top|left|right"))

    # Sparse direct solver
    solver = Preconditioner(a,"direct")

    # Solve the acoustic problem at t=0
    solvers.BVP(bf=a, lf=f, gf=sol, pre=solver, print=False)

    return sol, V


def compute_error(u, uex_cf, V):
    """Compute the relative L2 error ||u - uex|| / ||uex||."""
    mesh = V.mesh
    diff = u - uex_cf
    err  = sqrt(Integrate(InnerProduct(diff, diff), mesh))
    norm = sqrt(Integrate(InnerProduct(uex_cf, uex_cf), mesh))
    return np.real(err / norm)

### Sound Field in a Rectangular Room with Sound-hard Boundaries

The two-dimensional sound field in a rectangular room whose height is very small compared to the wavelength and with hard boundaries (Dirichlet boundary condition) is computed for the frequency $f=120$ Hz. The Dirichlet data is settled in such a manner that the exact solution is the real part of a plane wave
$$
\boldsymbol{U}_{ex}(\boldsymbol{x}, \omega)=(\cos\theta,\sin\theta)\exp(i(k_{0}x_0+k_{1}x_{1})),\qquad (k_0,k_1)=\frac{\omega}{c}(\cos\theta,\sin\theta),\qquad \theta=\frac{\pi}{4}.
$$

In [ ]:
# Physical data
freq = 120.
c    = 343.

# Exact solution: plane wave with wavenumber vector (k0, k1)
theta = np.pi / 4
k0    = 2 * np.pi * freq / c * np.cos(theta)
k1    = 2 * np.pi * freq / c * np.sin(theta)

# Build mesh (200 x 200 resolution equivalent)
mesh = make_rect_mesh(nx=50, ny=50)

# Define the exact solution as an NGSolve CoefficientFunction
uex = CoefficientFunction((
    np.cos(theta) * exp(1j * (k0 * x + k1 * y)),
    np.sin(theta) * exp(1j * (k0 * x + k1 * y))
))

# Compute FE solution
gfu, V = FEM_displacements(mesh, freq, uex, c)

# Compute and print the relative L2 error
print('L2-relative error:', compute_error(gfu, uex, V))

# Plot each component of the displacement field
Draw(gfu[0], mesh, animate_complex=True, height="3vh")

### Study of convergence

Let's check the order of convergence of this finite element discretization computing the error for sucesive refinements of the mesh

In [24]:
for nx in [25, 50, 100]:
    mesh = make_rect_mesh(nx=nx, ny=nx)
    u, V = FEM_displacements(mesh, freq, uex, c)

    print(f'Mesh size (approx) {min(5/nx, 4/nx):.6f}')
    print('L2-relative error:', compute_error(u, uex, V))

Mesh size (approx) 0.040000
L2-relative error: 0.00012815335218061672


### Exercise 
Perform a comparative study with respect to the pressure formulation: if a structured mesh (100x100) is fixedand use Lagrange finite elements (order $p$) for the pressure formulation, and Raviart-Thomas (order $p$) for the displacement formulation and vary $p=1,2,3$, which finite element approximation will provide a lower error? Which would be advisable in general terms: Raviart-Thomas or Lagrange discretizations?

In [25]:
## YOUR CODE HERE

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).